In [2]:
import numpy as np
def hex_to_dec(num):
    lookup = "ABCDEF"
    ans = 0
    for char in num:
        try:
            curr = int(char)
        except:
            curr =10+lookup.find(char)
        ans*=16
        ans+=curr
    return ans
def dec_to_bin(number: int, bits=-1):
    number=int(np.round(number, 0))
    neg=False
    out=""
    if (number<0):
        number*=-1
        number-=1
        neg=True
    while (number>0):
        res = number%2
        if (neg):
            res= 0 if (res==1) else 1
        out=f"{res}{out}"
        if (len(out)==(bits-1)):
            break
        number=int(number/2)
        
    if (neg):
        out=f"{1}{out}"
    else: out=f"{0}{out}"
    if (len(out)==0):
        out="0"
    while (len(out)<bits):
        out=f"{out[0]}{out}"
    return out
def dec_to_bin_unsigned(number: int, bits=-1):
    number=int(np.round(number, 0))
    out=""
    while (number>0):
        res = number%2
        out=f"{res}{out}"
        if (len(out)==(bits)):
            break
        number=int(number/2)
    if (len(out)==0):
        out="0"
    while (len(out)<bits):
        out=f"0{out}"
    return out
def bin_to_dec(number: str):
    output = 0
    neg = False
    if (number[0]=="1"):
        neg = True
    for i, char in enumerate(number[:0:-1]):
        if (neg):
            output+=(1-int(char))*(2**i)
        else:
            output+=int(char)*(2**i)
    if (neg):
        output*=-1
        output-=1
    return output
def hex_to_bin(num, width=0):
    ans = ""
    for char in num:
        ans+=dec_to_bin_unsigned(hex_to_dec(char),4)
    while (len(ans)<width):
        ans=f"0000{ans}"
    return ans[-width:]
def hex_to_int(num, width=0):
    ans = ""
    for char in num:
        ans+=dec_to_bin_unsigned(hex_to_dec(char),4)
    return bin_to_dec(ans[-width:])

In [13]:
import numpy as np
import os
import scipy.optimize as opt
import matplotlib.pyplot as plt
import re
def fit_eq(x,a):
    y = np.exp(x/(2**a))
    return y
def fit2(x,a):
     y=np.divide(1,(x/(2**a)))
     return y
eqs = [fit2, fit_eq]
matcher = re.compile(r"fixed_(\d+)_(\d+)")
pull_from = [6]
directories = [f"/home/caleb/sweeps/hls_argmax_{3*i-2}_{i}/p_prj/solution1/syn/verilog" for i in pull_from]
#directory="/home/caleb/sweeps/hls_fast_13_5/p_prj/solution1/syn/verilog"
# width=18
# nfrac=10
def plot_table(directory, precision):
    table = np.zeros(1024)
    # table_raw = np.zeros(1024)
    i=0
    width = precision[0]
    nfrac=  precision[0]-precision[1]
    files = os.listdir(directory)
    tables = []
    for file in files:
        if ".dat" in file:
            # if ("exp" in file):
                tables.append(file)
    
    bins = []
    table=[]

    current_table = tables[0]
    el_eq = eqs[1]
    with open(os.path.join(directory, current_table), "r") as f:
        current = f.readline()
        testbin = hex_to_bin(current[:-1],32)
        width = testbin.find("1")
        width=33-width
        # width=20
        while (len(current)>1):
            bins.append(f"{hex_to_bin(current[:-1],width)}\n")
            table.append(hex_to_int(current[:-1],width)/2**nfrac)
            i+=1
            current = f.readline()
    length=len(table)

    # test_x =  np.linspace(-(length-1),0, length)
    test_x =  np.linspace(-(length/2),(length/2-1), length)

    lookup_nfrac=0
    # test_x = test_x[::-1]
    #ab, bc = opt.curve_fit(el_eq, test_x, table)
    #lookup_nfrac=np.round(ab[0])
    overall_bits = int(matcher.findall(current_table)[0][0])
    decimal_bits=  int(matcher.findall(current_table)[0][1])
    print(2**(overall_bits-decimal_bits-1), -(length-1)//(2**lookup_nfrac), length, lookup_nfrac)

    # x_var = np.linspace(-(length-1)//(2**lookup_nfrac),0, length)
    # x_var = np.linspace(-(length/2)//(2**lookup_nfrac),(length/2-1)//(2**lookup_nfrac), length)
    x_var = np.linspace(-(length/2),(length/2-1), length)
    # x_var=x_var[::-1]
    
    # print(int(np.round(ab)))
    
    # for i in range(len(table)):
    #     if (table[i]==1):
    #         print(i)
    #         lookup_nfrac=np.log2(i)
    # lookup_nfrac = 2

    # x_var = np.linspace(-(length/2)//(2**lookup_nfrac),(length/2-1)//(2**lookup_nfrac), length)
    #x_var = np.linspace(0, (length-1)//(2**lookup_nfrac), length)
    
    with open("table_dump", "w") as f:
        f.writelines(bins)
    
    # for i, val in enumerate((table==1)):
    #     if (val):
    #         print(i)
    #print(np.max(table*2**nfrac)+1)
    plt.plot(x_var, np.concatenate((table[int(length/2):],table[0:int(length/2)])))
    # plt.plot(x_var, table, label=lookup_nfrac)
    plt.title(current_table)
    # plt.plot(x_var, np.divide(1, x_var))
    # plt.plot(x_var, np.exp(x_var))

for i in range(len(directories)):
    integers = pull_from[i]
    plot_table(directories[i],(18,8))
plt.xlim((-10,10))
plt.legend()
plt.show()

IndexError: list index out of range